# 02 Clean

Turns the raw Vektis and CBS files into clean tables. Every cleaning decision is written down: what I found, what I did, why.

In [ ]:
import pandas as pd

In [ ]:
vektis = pd.read_csv("../data/raw/Vektis Open Databestand Zorgverzekeringswet 2024 - gemeente.csv", sep=";")

In [ ]:
vektis.head()

## Step 1. Privacy bucket

In [ ]:
# Privacy bucket: groups under 10 people (hidden so no one can be identified)
# plus people with unknown sex, age or municipality. All lumped into one unlabelled row.
bucket = vektis[vektis['gemeentenaam'].isna()]
bucket

In [ ]:
bucket_people = bucket["aantal_bsn"].sum()
total_people = vektis["aantal_bsn"].sum()
share = bucket_people / total_people * 100
print(f"Bucket: {bucket_people:,.0f} people, {share:.1f}% of all people")

In [ ]:
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [ ]:
bucket_cost = bucket.filter(like="kosten").sum().sum()
cost_share = bucket_cost / total_costs * 100
print(f"Bucket: €{bucket_cost:,.0f}, {cost_share:.1f}% of all cost")

**Decision: privacy bucket**
- Found: one row with empty sex, age and municipality. Vektis puts groups under 10 people (privacy) and people with unknown sex, age or municipality here. It holds 412,394 people (2.2%) but only €167.8 million (0.3% of all cost).
- Did: removed the row. The raw file stays unchanged.
- Why: every KPI needs a municipality, and expected cost needs age and sex. This row has none of them, so it cannot be placed anywhere. Rejected: spreading it over municipalities, because we do not know where these people live.
- Effect: only 5 small cells are missing (Rozendaal, Schiermonnikoog, Vlieland). The bucket is cheap per person (about €760 per insured-year against €3,382), likely because many were insured for only part of the year. Removing it barely changes the national figures.

In [ ]:
vektis = vektis[vektis["gemeentenaam"].notna()]

In [ ]:
vektis.shape

In [ ]:
assert len(vektis) == 12991

## Step 2. Negative costs

In [ ]:
# Recreate after dropping the bucket, so costs and vektis have the same rows
costs = vektis.filter(like="kosten")
total_costs = costs.sum().sum()

In [ ]:
# How many negative cells?
neg_costs = costs < 0
neg_costs.sum().sum()

In [ ]:
# Which rows are they, and how big?
neg_rows = vektis[neg_costs.any(axis=1)]
neg_rows[["geslacht", "leeftijdsklasse", "gemeentenaam",
          "kosten_integrale_geboortezorg", "kosten_intramuraal_verblijf_ggz"]]

In [ ]:
row_total = costs.sum(axis=1)
town_total = row_total.groupby(vektis["gemeentenaam"]).sum()
town_total[["UITGEEST", "TWENTERAND"]]

**Decision: negative costs**
- Found: 2 negative cells out of about 338,000: Uitgeest, women 25-29, inpatient mental health stays (−€2,111), and Twenterand, women 30-34, birth care (−€2,300). Each is under 0.01% of that municipality's total cost.
- Did: kept them as they are.
- Why: they are real corrections (reversed or refunded claims), too small to move any result, and keeping them means totals match the official Vektis figures. Rejected: setting them to zero (totals would no longer match Vektis) and dropping the rows (would also remove 25 real cost cells per row).

## Step 3. Care abroad

In [ ]:
abroad = vektis["kosten_grensoverschrijdende_zorg"]
abroad_costs = abroad.sum()
abroad_share = abroad_costs / total_costs * 100
print(f"Care abroad: €{abroad_costs:,.0f}, {abroad_share:.1f}% of all cost")

In [ ]:
abroad_town = vektis["kosten_grensoverschrijdende_zorg"].groupby(vektis["gemeentenaam"]).sum()
abroad_pct = abroad_town / town_total * 100
abroad_pct.sort_values(ascending=False)

In [ ]:
abroad_pct.median()

**Decision: care abroad**
- Found: care abroad (`kosten_grensoverschrijdende_zorg`) is €347.3 million, 0.6% of all cost, and the leaflet says it is incomplete because foreign bills arrive late. It is concentrated in border municipalities: median 0.4% of a municipality's cost, but Sluis 18%, Hulst 10%, Terneuzen 8% (Zeeuws-Vlaanderen, next to Belgium) and Vaals 5% (next to Germany and Belgium).
- Did: kept the column.
- Why: for border towns, care abroad is part of their normal hospital care. Excluding it would make Sluis look about 18% cheaper than it is and give the purchasing team a false signal. Rejected: excluding the column.
- Limitation: because the column is incomplete, border towns' true cost is probably higher than shown. This goes in the README's Limitations.